In [1]:
pip install fastapi uvicorn pypdf langchain langchain-community langchain-openai faiss-cpu python-multipart pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import io
import uuid
from typing import List, Dict, Any, Optional
from fastapi import FastAPI, UploadFile, File, HTTPException, status, Query
from pydantic import BaseModel, Field
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# Initialize FastAPI App
app = FastAPI(
    title="RAG API Engine",
    description="API for document processing, chunking, indexing, and context-based question answering.",
    version="1.0.0"
)
# Set OpenAI API Key (or set via environment variable: export OPENAI_API_KEY="your-key")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "your-openai-api-key-here")

# In-Memory FAISS Vector Store Initialization
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store: Optional[FAISS] = None

# --- Pydantic Data Models ---
class IngestResponse(BaseModel):
    message: str
    filename: str
    file_id: str
    total_chunks: int
    extracted_character_count: int

class QueryRequest(BaseModel):
    question: str = Field(
        ..., 
        json_schema_extra={"example": "What are the key findings mentioned in the uploaded report?"}
    )
class QueryResponse(BaseModel):
    question: str
    answer: str
    sources: List[Dict[str, Any]]

# --- Core Helper Functions ---
def parse_pdf(file_bytes: bytes) -> str:
    """Extracts text page by page from raw PDF bytes."""
    try:
        reader = PdfReader(io.BytesIO(file_bytes))
        extracted_text = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                extracted_text.append(f"[Page {i+1}]\n{text}")
        return "\n\n".join(extracted_text)
    except Exception as e:
        raise ValueError(f"Failed to parse PDF document: {str(e)}")

def chunk_text(text: str, source_filename: str, file_id: str) -> List[Document]:
    """Splits text into chunks with rich metadata attached."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", " ", ""]
    )
    docs = splitter.create_documents(
        texts=[text],
        metadatas=[{
            "filename": source_filename,
            "file_id": file_id,
            "total_length": len(text)
        }]
    )
    return docs

# --- API Endpoints ---

@app.get("/health", tags=["Health"])
async def health_check():
    """Health check endpoint to verify server status."""
    return {"status": "ok", "vector_store_initialized": vector_store is not None}

@app.post("/upload", status_code=status.HTTP_201_CREATED, tags=["RAG Core"])
async def upload_document(file: UploadFile = File(...)):
    # File validation...
    if not file.filename.endswith(".pdf"):
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Unsupported File Format. Only PDF files (.pdf) are allowed."
        )
    # 2. File Reading & Empty File Validation
    content = await file.read()
    if len(content) == 0:
        raise HTTPException(
            status_code=Status.HTTP_400_BAD_REQUEST,
            detail="Uploaded file is empty (0 bytes)."
        )

    # 3. File Size Validation (Max Limit: 10MB)
    MAX_FILE_SIZE = 10 * 1024 * 1024  # 10 MB
    if len(content) > MAX_FILE_SIZE:
        raise HTTPException(
            status_code=Status.HTTP_413_REQUEST_ENTITY_TOO_LARGE,
            detail="File size exceeds maximum allowed limit of 10MB."
        )

    # 4. Text Extraction
    try:
        raw_text = parse_pdf(content)
    except Exception as e:
        raise HTTPException(
            status_code=Status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail=f"Corrupted or invalid PDF content: {str(e)}"
        )

    if not raw_text.strip():
        raise HTTPException(
            status_code=Status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="PDF contains no extractable text. It may contain scanned images without OCR."
        )

    # 5. Document Chunking & Metadata Injection
    file_id = str(uuid.uuid4())
    chunks = chunk_text(raw_text, file.filename, file_id)

    # 6. Embedding & Vector Database Ingestion
    if vector_store is None:
        vector_store = FAISS.from_documents(chunks, embeddings)
    else:
        vector_store.add_documents(chunks)

    return IngestResponse(
        message="Document successfully processed and indexed into vector database.",
        filename=file.filename,
        file_id=file_id,
        total_chunks=len(chunks),
        extracted_character_count=len(raw_text)
    )

@app.post("/query", response_model=QueryResponse, tags=["RAG Core"])
async def query_rag(payload: QueryRequest):
    """
    Retrieves contextually relevant text chunks and generates grounded answers via LLM.
    """
    global vector_store

    if vector_store is None:
        raise HTTPException(
            status_code=Status.HTTP_400_BAD_REQUEST,
            detail="No documents have been uploaded or indexed yet. Please upload a PDF first."
        )

    if not payload.question.strip():
        raise HTTPException(
            status_code=Status.HTTP_400_BAD_REQUEST,
            detail="Question query cannot be empty."
        )

    # Setup RAG Chain
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    prompt = ChatPromptTemplate.from_template("""
    You are an intelligent assistant. Answer the question using ONLY the provided context.
    If the answer cannot be found in the context, state clearly that the answer is not present in the document.

    Context:
    {context}

    Question:
    {input}
    """)

    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)

    # Execute Chain
    response = rag_chain.invoke({"input": payload.question})

    # Prepare Sources Metadata Response
    sources_metadata = [
        {
            "filename": doc.metadata.get("filename", "unknown"),
            "file_id": doc.metadata.get("file_id", "N/A"),
            "snippet": doc.page_content[:150] + "..."
        }
        for doc in response.get("context", [])
    ]

    return QueryResponse(
        question=payload.question,
        answer=response["answer"],
        sources=sources_metadata
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=True)

C:\Users\prati\AppData\Local\Temp\ipykernel_16520\1800591011.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
INFO:     Will watch for changes in these directories: ['e:\\Linkfici(Intership)\\Linkific Tasks\\Day17']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [16520] using WatchFiles
INFO:     Stopping reloader process [16520]
